# Task 2: Graph Modeling & Centrality Analysis
In this notebook, we perform community detection, centrality metrics calculation, and core-periphery structure identification for each subreddit from the collected interactions data.


In [1]:
import pandas as pd
import networkx as nx
import numpy as np
import os
import community.community_louvain as community_louvain # pip install python-louvain
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [2]:
print("Loading data...")
edges_df = pd.read_csv('graphs/all_edges_consolidated.csv')

# Aggregate the weights over all time windows for each subreddit
global_edges = edges_df.groupby(['Source', 'Target', 'Subreddit'], as_index=False)['Weight'].sum()

# Get the list of subreddits to process
subreddits = global_edges['Subreddit'].unique()
print(f"Found {len(subreddits)} subreddits.")
global_edges.head()

Loading data...
Found 20 subreddits.


,Source,Target,Subreddit,Weight
0,-----Kyle-----,OhWalter,ClimateOffensive,1.29
1,---M0NK---,ChemicalLight4133,climatechange,2.99
2,---M0NK---,Laker4Life9,climatechange,3.92
3,---M0NK---,LyraSerpentine,climatechange,2.20
4,---M0NK---,PhilHallUSA,climatechange,2.97


In [3]:
def analyze_subreddit_graph(edges_group):
    # Construct a Directed Graph
    G_directed = nx.DiGraph()
    for _, row in edges_group.iterrows():
        G_directed.add_edge(row['Source'], row['Target'], weight=row['Weight'])
        
    # Remove self-loops
    G_directed.remove_edges_from(nx.selfloop_edges(G_directed))
    
    # 1. Centrality Metrics
    # Betweenness Centrality (using an unweighted approximation or ignoring weights for speed)
    # For large graphs, k=50 or 100 might be needed for approximation
    k = min(100, len(G_directed.nodes())) if len(G_directed.nodes()) > 500 else None
    betweenness = nx.betweenness_centrality(G_directed, k=k, weight='weight' if k is None else None)
    
    # PageRank
    try:
        pagerank = nx.pagerank(G_directed, weight='weight')
    except:
        pagerank = {node: 0 for node in G_directed.nodes()}
        
    # Construct an Undirected Graph for Louvain and Core-Periphery
    G_undirected = G_directed.to_undirected()
    
    # 2. Community Detection (Louvain)
    try:
        louvain_partition = community_louvain.best_partition(G_undirected, weight='weight')
    except:
        louvain_partition = {node: -1 for node in G_undirected.nodes()}
        
    # 3. Core-Periphery (k-core)
    core_numbers = nx.core_number(G_undirected)
    
    # Assemble into a DataFrame
    metrics = []
    for node in G_directed.nodes():
        metrics.append({
            'User': node,
            'Betweenness_Centrality': betweenness.get(node, 0),
            'PageRank': pagerank.get(node, 0),
            'Community_ID': louvain_partition.get(node, -1),
            'Core_Number': core_numbers.get(node, 0),
            'Degree': G_directed.degree(node),
            'In_Degree': G_directed.in_degree(node),
            'Out_Degree': G_directed.out_degree(node)
        })
        
    metrics_df = pd.DataFrame(metrics)
    return metrics_df, G_directed

In [6]:
all_metrics = []

for subreddit in subreddits:
    print(f"Processing Subreddit: {subreddit}...")
    edges_group = global_edges[global_edges['Subreddit'] == subreddit]
    
    # Skip extremely small networks
    if len(edges_group) < 5:
        print("  - Not enough edges, skipping.")
        continue
        
    metrics_df, G_directed = analyze_subreddit_graph(edges_group)
    metrics_df['Subreddit'] = subreddit
    all_metrics.append(metrics_df)
    
final_metrics_df = pd.concat(all_metrics, ignore_index=True)

# Save to CSV
if not os.path.exists('graphs'):
    os.makedirs('graphs')
final_metrics_df.to_csv('graphs/node_metrics.csv', index=False)
print("Finished processing all subreddits and saved to 'graphs/node_metrics.csv'.")
final_metrics_df.head()

Processing Subreddit: ClimateOffensive...
Processing Subreddit: climatechange...
Processing Subreddit: climateskeptics...
Processing Subreddit: ClimateShitposting...
Processing Subreddit: ExtinctionRebellion...
Processing Subreddit: ClimateActionPlan...
Processing Subreddit: Green...
Processing Subreddit: Climate_Nuremberg...
Processing Subreddit: ClimateNews...
Processing Subreddit: ecologie...
Processing Subreddit: SunriseMovement...
Processing Subreddit: climatesolutions...
Processing Subreddit: ClimateCrisisCanada...
Processing Subreddit: Ecologisme...
Processing Subreddit: FridaysForFuture...
Processing Subreddit: climate_discussion...
Processing Subreddit: ClimateCO...
Processing Subreddit: climatepolicy...
Processing Subreddit: greenpeace...
Processing Subreddit: climatejustice...
Finished processing all subreddits and saved to 'graphs/node_metrics.csv'.


,User,Betweenness_Centrality,PageRank,Community_ID,Core_Number,Degree,In_Degree,Out_Degree,Subreddit
0,-----Kyle-----,0.0,0.000027,0,1,1,0,1,ClimateOffensive
1,OhWalter,0.0,0.000052,0,2,3,2,1,ClimateOffensive
2,---rayne---,0.0,0.000072,1,1,3,2,1,ClimateOffensive
3,harold__hadrada,0.0,0.001839,1,6,61,61,0,ClimateOffensive
4,--Blaise--,0.0,0.000027,2,1,1,0,1,ClimateOffensive


In [7]:
# Let's verify by finding the top 5 influencers (PageRank) per subreddit
top_influencers = final_metrics_df.sort_values(
    ['Subreddit', 'PageRank'], ascending=[True, False]
).groupby('Subreddit').head(5)

print("Top 5 users by PageRank in some subreddits:")
top_influencers[['Subreddit', 'User', 'PageRank', 'Betweenness_Centrality', 'Community_ID', 'Core_Number']].head(15)

Top 5 users by PageRank in some subreddits:


,Subreddit,User,PageRank,Betweenness_Centrality,Community_ID,Core_Number
39228,ClimateActionPlan,exprtcar,0.034579,0.160495,6,8
39211,ClimateActionPlan,thespaceageisnow,0.022440,0.034632,4,8
39215,ClimateActionPlan,WaywardPatriot,0.013713,0.089922,5,8
39386,ClimateActionPlan,Falom,0.011563,0.071594,4,8
39334,ClimateActionPlan,coolbern,0.008892,0.010876,12,8
56086,ClimateCO,wasachrozine,0.081055,0.114628,5,3
56114,ClimateCO,EmBejarano,0.057314,0.057271,4,3
56116,ClimateCO,CurlyHairedFuk,0.053112,0.069519,4,2
56100,ClimateCO,aless4ndra,0.051866,0.001466,3,2
56112,ClimateCO,bluntforce21,0.050835,0.000173,3,2
